# Text-to-Speech Seq2Seq Model Training
This notebook trains a Seq2Seq model with Bahdanau attention on the LJSpeech dataset.

In [ ]:
# Cell 1: Setup & GPU Check
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

!pip install torch librosa g2p_en soundfile scipy pandas tqdm matplotlib


In [ ]:
# Cell 2: Clone repo or copy source code
import os
import sys

if not os.path.exists('ML-miniproject'):
    !git clone https://github.com/Roshan2607/ML-miniproject

sys.path.append(os.path.abspath('ML-miniproject'))


In [ ]:
# Cell 3: Download LJSpeech dataset
import os

if not os.path.exists('LJSpeech-1.1'):
    !wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xjf LJSpeech-1.1.tar.bz2

import glob
wav_files = glob.glob('LJSpeech-1.1/wavs/*.wav')
print(f"Found {len(wav_files)} wav files.")


In [ ]:
# Cell 4: Preprocess in batches
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.preprocessing import audio_to_spectrogram
from src.config import Config

# Update config to point to Colab paths
Config.RAW_DATA_DIR = 'LJSpeech-1.1'
Config.PROCESSED_DATA_DIR = 'processed_data'

os.makedirs(Config.PROCESSED_DATA_DIR, exist_ok=True)

metadata_path = os.path.join(Config.RAW_DATA_DIR, 'metadata.csv')
metadata = pd.read_csv(metadata_path, sep='|', header=None, 
                       names=['id', 'transcription', 'normalized_transcription'])
metadata.dropna(inplace=True)

BATCH_SIZE_PREP = 500
wav_dir = os.path.join(Config.RAW_DATA_DIR, 'wavs')

print("Starting preprocessing...")
for i in tqdm(range(0, len(metadata), BATCH_SIZE_PREP)):
    batch = metadata.iloc[i:i+BATCH_SIZE_PREP]
    for _, row in batch.iterrows():
        audio_id = row['id']
        wav_path = os.path.join(wav_dir, f"{audio_id}.wav")
        spec_path = os.path.join(Config.PROCESSED_DATA_DIR, f"{audio_id}.npy")
        if not os.path.exists(spec_path) and os.path.exists(wav_path):
            mel_spec = audio_to_spectrogram(wav_path, 
                                            n_fft=Config.n_fft, 
                                            hop_length=Config.hop_length, 
                                            win_length=Config.win_length, 
                                            preemph=Config.preemphasis_k)
            np.save(spec_path, mel_spec)
print("Preprocessing completed.")


In [ ]:
# Cell 5: Training Configuration
import torch

MAX_SAMPLES = 2000 # configurable, user can increase
BATCH_SIZE = 16
NUM_EPOCHS = 25
LEARNING_RATE = 0.001
TEACHER_FORCING_RATIO = 0.5
TRAIN_SPLIT = 0.9

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Cell 6: Create DataLoaders
from torch.utils.data import DataLoader, random_split
from src.dataset import LJSpeechDataset, collate_fn

dataset = LJSpeechDataset(metadata_path=metadata_path, processed_dir=Config.PROCESSED_DATA_DIR)

if len(dataset) > MAX_SAMPLES:
    indices = list(range(MAX_SAMPLES))
    dataset = torch.utils.data.Subset(dataset, indices)

train_size = int(TRAIN_SPLIT * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


In [ ]:
# Cell 7: Initialize Model
import torch.nn as nn
from src.model import Seq2SeqTTS
from src.preprocessing import get_vocab_size

vocab_size = get_vocab_size()

model = Seq2SeqTTS(
    vocab_size=vocab_size,
    embed_dim=Config.embed_dim,
    hidden_dim=Config.hidden_dim,
    output_dim=Config.spec_bins
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {param_count:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
# Cell 8: Training Loop
import json
import os

def masked_mse_loss(prediction, target, lengths):
    batch_size, max_time, _ = target.shape
    time_indices = torch.arange(max_time, device=target.device).unsqueeze(0)
    mask = (time_indices < lengths.unsqueeze(1))
    frame_loss = torch.mean((prediction - target) ** 2, dim=2)
    frame_loss = frame_loss * mask.float()
    denominator = mask.float().sum().clamp(min=1.0)
    return frame_loss.sum() / denominator

CHECKPOINT_DIR = '/content/drive/MyDrive/tts_checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    for text, seq_lengths, target_spec, spec_lengths in train_loader:
        text, seq_lengths = text.to(device), seq_lengths.to(device)
        target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
        
        optimizer.zero_grad()
        prediction = model(text, seq_lengths, target_spec, TEACHER_FORCING_RATIO)
        loss = masked_mse_loss(prediction, target_spec, spec_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_train_loss += loss.item()
        
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for text, seq_lengths, target_spec, spec_lengths in val_loader:
            text, seq_lengths = text.to(device), seq_lengths.to(device)
            target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
            
            prediction = model(text, seq_lengths, target_spec, teacher_forcing_ratio=0.0)
            loss = masked_mse_loss(prediction, target_spec, spec_lengths)
            epoch_val_loss += loss.item()
            
    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss
        }, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        print("Saved new best model.")

with open(os.path.join(CHECKPOINT_DIR, 'loss_history.json'), 'w') as f:
    json.dump({'train_loss': train_losses, 'val_loss': val_losses}, f)


In [ ]:
# Cell 9: Plot Training Curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, label='Train Loss')
plt.plot(range(1, NUM_EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Masked MSE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Cell 10: Inference & Listen
import IPython.display as ipd
import soundfile as sf
from src.preprocessing import text_to_sequence, spectrogram_to_audio

checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

input_text = "Hello, this is a test of the text to speech system."
print(f"Input text: {input_text}")

seq = text_to_sequence(input_text)
seq_tensor = torch.LongTensor(seq).unsqueeze(0).to(device)
seq_length = torch.LongTensor([len(seq)]).to(device)

with torch.no_grad():
    dummy_target = torch.zeros(1, 400, Config.spec_bins).to(device) # Provide a dummy target length
    prediction = model(seq_tensor, seq_length, target_spectrogram=dummy_target, teacher_forcing_ratio=0.0)
    
pred_spec = prediction.squeeze(0).cpu().numpy()
generated_audio = spectrogram_to_audio(pred_spec, 
                                       n_fft=Config.n_fft, 
                                       hop_length=Config.hop_length, 
                                       win_length=Config.win_length, 
                                       preemph=Config.preemphasis_k)

output_wav_path = 'generated_sample.wav'
sf.write(output_wav_path, generated_audio, Config.sample_rate)

ipd.Audio(output_wav_path)


In [ ]:
# Cell 11: Save everything to Drive
import shutil

drive_save_dir = '/content/drive/MyDrive/tts_outputs/'
os.makedirs(drive_save_dir, exist_ok=True)

shutil.copy(output_wav_path, drive_save_dir)
print(f"Saved {output_wav_path} to {drive_save_dir}")
